In [ ]:
import os
import warnings
import logging

# ===== SILENCE ALL WARNINGS =====
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

# Suppress Python warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Suppress TensorFlow logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

# Now import
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/temporal-modelling/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

# TensorFlow imports with suppressed logging
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV

# Final suppression of retracing warnings (these are annoying but harmless)
tf.keras.backend.clear_session()

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_run_name = 'rnn_v2'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report   = False
save_model  = False
random_search = False

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

In [ ]:
num_pattern  = 'acc|rot|thm|tof'

# In your notebook, after defining columns
raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns,
    rot_cols=rot_columns,
    rotation_mode="delta_euler",   # or "euler"
    thm_cols=thm_columns,
    tof_cols=None,                 # auto-detects all 320
    tof_mode="raw",                # change to "baseline" if you want light stats
)

preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), make_column_selector(pattern="acc|rot|thm|tof")),
], remainder="drop", verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

sequence_builder = utils.SequencePadder(maxlen=60, padding_value=-999.0)  # ← your new window size

rnn_clf = utils.KerasRNNClassifier(
    rnn_type="lstm",
    rnn_units=(128, 64),
    dense_units=(64,),
    dropout=0.2,
    bidirectional=True,
    learning_rate=5e-4,
    batch_size=16,
    epochs=150,
    patience=20,
    class_weight_mode="balanced",
)

classifier = utils.ManyToOneWrapperRNN(estimator=rnn_clf, target="gesture_action")

pipeline = Pipeline([
    ("raw_extractor", raw_extractor),
    ("preprocessor", preprocessor),
    ("sequence_builder", sequence_builder),
    ("classifier", classifier),
])

cv = GroupKFold(n_splits=3)

param_grid = {
    # RawSequenceExtractor params
    'raw_extractor__rotation_mode': ['delta_euler'],
    'raw_extractor__acc_cols': [acc_columns],
    'raw_extractor__rot_cols': [rot_columns],
    'raw_extractor__thm_cols': [thm_columns],

    # SequencePadder params
    'sequence_builder__maxlen': [10],
    'sequence_builder__padding_value': [0.0],

    # RNN params (nested under classifier__estimator__)
    'classifier__estimator__rnn_type': ['rnn'],
    'classifier__estimator__rnn_units': [(256, 128, 64)],
    'classifier__estimator__dense_units': [(128, 64)],
    'classifier__estimator__dropout': [0.1],
    'classifier__estimator__learning_rate': [1e-3],
    'classifier__estimator__batch_size': [16],
    'classifier__estimator__epochs': [200],
    'classifier__estimator__patience': [10],
    "classifier__estimator__bidirectional": [True],
    "classifier__estimator__class_weight_mode": ["balanced"],
}

In [ ]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,  # LSTM is slow, keep iterations low
                cv=cv,
                random_state=42,
                n_jobs=1,              # safer with tensorflow/keras on Kaggle
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,              # safer with tensorflow/keras on Kaggle
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']
        
        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
    
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

In [ ]:
master_cv_results_df

In [ ]:
if do_report:

    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'lstm_v1_per_gesture_scores.csv')

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.class_weight import compute_class_weight

# Suppress warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

print("Loading data...")
# Load your data (adjust path as needed)
raw_train_df = pd.read_csv('data/train.csv')
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()
target_only_df['gesture_action'] = target_only_df['gesture'].str.split(' - ').str[-1]

# Define columns
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

# Take a smaller sample for testing
print("Sampling data...")
unique_seqs = target_only_df['sequence_id'].unique()[:100]  # Use 100 sequences for testing
train_df_sample = target_only_df[target_only_df['sequence_id'].isin(unique_seqs)].copy()

print(f"Data shape: {train_df_sample.shape}")
print(f"Unique sequences: {train_df_sample['sequence_id'].nunique()}")
print(f"Unique gestures: {train_df_sample['gesture_action'].nunique()}")
print(f"Gesture distribution:\n{train_df_sample['gesture_action'].value_counts()}")

# ===================== SIMPLE SEQUENCE BUILDER =====================
def build_sequences(df, maxlen=60):
    """Convert raw data to sequences"""
    sequences = []
    labels = []
    seq_ids = []
    
    for seq_id, group in df.groupby('sequence_id'):
        # Get sensor data
        acc = group[acc_columns].values
        rot = group[rot_columns].values
        thm = group[thm_columns].values
        
        # Combine all sensors
        combined = np.hstack([acc, rot, thm])
        
        # Pad or truncate
        if len(combined) > maxlen:
            combined = combined[:maxlen]
        else:
            pad_len = maxlen - len(combined)
            combined = np.vstack([combined, np.zeros((pad_len, combined.shape[1]))])
        
        sequences.append(combined)
        labels.append(group['gesture_action'].iloc[0])
        seq_ids.append(seq_id)
    
    return np.array(sequences), np.array(labels), seq_ids

# Build sequences
print("\nBuilding sequences...")
X, y, seq_ids = build_sequences(train_df_sample, maxlen=60)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Unique labels: {np.unique(y)}")

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Encoded labels: {np.unique(y_encoded)}")
print(f"Number of classes: {len(le.classes_)}")

# ===================== SIMPLE RNN MODEL =====================
def create_model(input_shape, num_classes, rnn_units=64, dropout=0.2):
    model = keras.Sequential([
        layers.Masking(mask_value=0.0, input_shape=input_shape),
        layers.LSTM(rnn_units, dropout=dropout, return_sequences=False),
        layers.Dense(32, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ===================== SKLEARN WRAPPER =====================
class KerasSequenceClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, rnn_units=64, dropout=0.2, epochs=50, batch_size=16, patience=10):
        self.rnn_units = rnn_units
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.model = None
        self.label_encoder = None
        
    def fit(self, X, y):
        # Encode labels
        self.label_encoder = LabelEncoder()
        y_enc = self.label_encoder.fit_transform(y)
        
        # Create model
        input_shape = (X.shape[1], X.shape[2])
        num_classes = len(self.label_encoder.classes_)
        self.model = create_model(input_shape, num_classes, self.rnn_units, self.dropout)
        
        # Early stopping
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=self.patience, 
            restore_best_weights=True
        )
        
        # Train
        self.model.fit(
            X, y_enc,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=0
        )
        return self
    
    def predict(self, X):
        proba = self.model.predict(X, verbose=0)
        return self.label_encoder.inverse_transform(np.argmax(proba, axis=1))
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

# ===================== RUN CROSS-VALIDATION =====================
print("\n" + "="*50)
print("Running Cross-Validation")
print("="*50)

# Use GroupKFold to keep sequences together
groups = seq_ids
cv = GroupKFold(n_splits=3)

# Test different architectures
architectures = [
    {'rnn_units': 32, 'dropout': 0.2},
    {'rnn_units': 64, 'dropout': 0.2},
    {'rnn_units': 128, 'dropout': 0.2},
    {'rnn_units': 64, 'dropout': 0.3},
]

results = []
for arch in architectures:
    print(f"\nTesting: rnn_units={arch['rnn_units']}, dropout={arch['dropout']}")
    
    clf = KerasSequenceClassifier(
        rnn_units=arch['rnn_units'],
        dropout=arch['dropout'],
        epochs=50,
        batch_size=16,
        patience=10
    )
    
    try:
        scores = cross_val_score(clf, X, y, cv=cv, groups=groups, scoring='accuracy', n_jobs=1)
        print(f"  Scores: {scores}")
        print(f"  Mean: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")
        results.append({
            'rnn_units': arch['rnn_units'],
            'dropout': arch['dropout'],
            'mean_score': scores.mean(),
            'std_score': scores.std(),
            'scores': scores
        })
    except Exception as e:
        print(f"  Error: {e}")

# Print summary
print("\n" + "="*50)
print("SUMMARY RESULTS")
print("="*50)
for r in results:
    print(f"rnn_units={r['rnn_units']}, dropout={r['dropout']}: {r['mean_score']:.4f} +/- {r['std_score']:.4f}")

# Train final model on all data
print("\n" + "="*50)
print("Training final model on all data")
print("="*50)

best_arch = max(results, key=lambda x: x['mean_score']) if results else architectures[0]
print(f"Using best architecture: rnn_units={best_arch['rnn_units']}, dropout={best_arch['dropout']}")

final_model = KerasSequenceClassifier(
    rnn_units=best_arch['rnn_units'],
    dropout=best_arch['dropout'],
    epochs=100,
    batch_size=16,
    patience=15
)
final_model.fit(X, y)

# Evaluate on training (just to see if it works)
train_score = final_model.score(X, y)
print(f"Final model training accuracy: {train_score:.4f}")

print("\n✅ Done! No NaN scores.")